In [1]:
import numpy as np
import pandas as pd

In [2]:
avg_volumes_df = pd.read_csv('facility_volume.csv')

In [3]:
display(avg_volumes_df)

,facility_name,Volume MD1,Volume MD2,Volume PM
0,AHG,1.23,4.2,0.56
1,AHM,0.7,0.97,0.3
2,AMMA,NC,NC,NC
3,ASFC,NC,NC,NC
4,ASJC,NC,NC,NC
...,...,...,...,...
130,NHD,0.125,0.125,0.6
131,NHC,0.125,0.125,1
132,NHA,0.01,0.01,1
133,NHF,0.01,0.01,1.5


In [4]:
n_days = 365
start_date = pd.Timestamp("2025-01-01")

In [5]:
col_map = {
    "facility_name": "facilities",
    "Volume MD1": "avg_md1",
    "Volume MD2": "avg_md2",
    "Volume PM": "avg_pm",
}

avg_volumes_df = avg_volumes_df.rename(columns=col_map)

display(avg_volumes_df)

,facilities,avg_md1,avg_md2,avg_pm
0,AHG,1.23,4.2,0.56
1,AHM,0.7,0.97,0.3
2,AMMA,NC,NC,NC
3,ASFC,NC,NC,NC
4,ASJC,NC,NC,NC
...,...,...,...,...
130,NHD,0.125,0.125,0.6
131,NHC,0.125,0.125,1
132,NHA,0.01,0.01,1
133,NHF,0.01,0.01,1.5


In [6]:
for c in ["avg_md1", "avg_md2", "avg_pm"]:
    avg_volumes_df[c] = pd.to_numeric(avg_volumes_df[c].replace("NC", np.nan), errors="coerce")
    
display(avg_volumes_df)

,facilities,avg_md1,avg_md2,avg_pm
0,AHG,1.230,4.200,0.56
1,AHM,0.700,0.970,0.30
2,AMMA,NaN,NaN,NaN
3,ASFC,NaN,NaN,NaN
4,ASJC,NaN,NaN,NaN
...,...,...,...,...
130,NHD,0.125,0.125,0.60
131,NHC,0.125,0.125,1.00
132,NHA,0.010,0.010,1.00
133,NHF,0.010,0.010,1.50


In [ ]:
# avg_volumes_df = avg_volumes_df.set_index('facility')
# avg_volumes_df.index.is_unique

print(avg_volumes_df)
avg_volumes_df.to_csv('output/cleaned_average_volume.csv')

    facilities  avg_md1  avg_md2  avg_pm
0          AHG    1.230    4.200    0.56
1          AHM    0.700    0.970    0.30
2         AMMA      NaN      NaN     NaN
3         ASFC      NaN      NaN     NaN
4         ASJC      NaN      NaN     NaN
..         ...      ...      ...     ...
130        NHD    0.125    0.125    0.60
131        NHC    0.125    0.125    1.00
132        NHA    0.010    0.010    1.00
133        NHF    0.010    0.010    1.50
134        NHG    0.010    0.010    2.30

[135 rows x 4 columns]


In [ ]:
cleaned_avg_volumes_df = pd.read_csv('output/cleaned_average_volume.csv')
display(cleaned_avg_volumes_df)

,Unnamed: 0,facilities,avg_md1,avg_md2,avg_pm
0,0,AHG,1.230,4.200,0.56
1,1,AHM,0.700,0.970,0.30
2,2,AMMA,NaN,NaN,NaN
3,3,ASFC,NaN,NaN,NaN
4,4,ASJC,NaN,NaN,NaN
...,...,...,...,...,...
130,130,NHD,0.125,0.125,0.60
131,131,NHC,0.125,0.125,1.00
132,132,NHA,0.010,0.010,1.00
133,133,NHF,0.010,0.010,1.50


In [9]:
# skip facilities/shifts with NaN

keep = (cleaned_avg_volumes_df[['avg_md1','avg_md2','avg_pm']].fillna(0) > 0).any(axis=1)
cleaned_avg_volumes_df = cleaned_avg_volumes_df[keep]
display(cleaned_avg_volumes_df)

,Unnamed: 0,facilities,avg_md1,avg_md2,avg_pm
0,0,AHG,1.230,4.200,0.56
1,1,AHM,0.700,0.970,0.30
9,9,BHBRMC,NaN,NaN,0.07
10,10,BHDCHV,0.030,NaN,0.03
11,11,BHFWCH,0.070,0.630,0.23
...,...,...,...,...,...
130,130,NHD,0.125,0.125,0.60
131,131,NHC,0.125,0.125,1.00
132,132,NHA,0.010,0.010,1.00
133,133,NHF,0.010,0.010,1.50


In [10]:
synthetic_data = []
rng = np.random.default_rng(123)


for row in cleaned_avg_volumes_df.itertuples(index=False):
    facility = row.facilities
    avg_md1  = float(row.avg_md1 or 0.0)
    avg_md2  = float(row.avg_md2 or 0.0)
    avg_pm   = float(row.avg_pm  or 0.0)

    for day in range(n_days):
        date = start_date + pd.Timedelta(days=day)
        seasonal_factor = 1 + 0.2 * np.sin(2 * np.pi * day / 365)
        
#         print(pd.isnull(avg_md1))
        
        if pd.isnull(avg_md1):
            md1 = float("nan")
        else:
            md1 = rng.poisson(lam=max(avg_md1 * seasonal_factor, 0))
            
        if pd.isnull(avg_md2):
            md2 = float("nan")
        else:
            md2 = rng.poisson(lam=max(avg_md2 * seasonal_factor, 0))
            
        if pd.isnull(avg_pm):
            pm = float("nan")
        else:
            pm = rng.poisson(lam=max(avg_pm * seasonal_factor, 0))

#         md1 = rng.poisson(lam=max(avg_md1 * seasonal_factor, 0))
#         md2 = rng.poisson(lam=max(avg_md2 * seasonal_factor, 0))
#         pm  = rng.poisson(lam=max(avg_pm  * seasonal_factor, 0))

        synthetic_data.append([facility, date, md1, md2, pm])

synthetic_df = pd.DataFrame(synthetic_data, columns=["facility", "date", "md1", "md2", "pm"])
display(synthetic_df)

,facility,date,md1,md2,pm
0,AHG,2025-01-01,1.0,2.0,2.0
1,AHG,2025-01-02,3.0,6.0,0.0
2,AHG,2025-01-03,0.0,3.0,0.0
3,AHG,2025-01-04,3.0,8.0,0.0
4,AHG,2025-01-05,0.0,4.0,0.0
...,...,...,...,...,...
44160,NHG,2025-12-27,1.0,0.0,2.0
44161,NHG,2025-12-28,0.0,0.0,3.0
44162,NHG,2025-12-29,0.0,0.0,0.0
44163,NHG,2025-12-30,0.0,0.0,6.0


In [ ]:
synthetic_df.to_csv("output/synthetic_data.csv")

In [12]:
# Ensure proper dtypes
synthetic_df['date'] = pd.to_datetime(synthetic_df['date'])
# Keep only the columns we need and melt to long format for convenience
value_cols = ['md1', 'md2', 'pm']
df_long = synthetic_df.melt(id_vars=['facility', 'date'], value_vars=value_cols,
                  var_name='shift', value_name='volume')
# Sort for sanity
df_long = df_long.sort_values(['facility', 'shift', 'date']).reset_index(drop=True)

In [13]:
display(df_long)

,facility,date,shift,volume
0,AHG,2025-01-01,md1,1.0
1,AHG,2025-01-02,md1,3.0
2,AHG,2025-01-03,md1,0.0
3,AHG,2025-01-04,md1,3.0
4,AHG,2025-01-05,md1,0.0
...,...,...,...,...
132490,wplex,2025-12-27,pm,0.0
132491,wplex,2025-12-28,pm,0.0
132492,wplex,2025-12-29,pm,0.0
132493,wplex,2025-12-30,pm,0.0


In [14]:
import os
import matplotlib.pyplot as plt


os.makedirs("figs", exist_ok=True)

for facility, g in df_long.groupby('facility'):
    plt.figure(figsize=(12, 5))
    for shift, gs in g.groupby('shift'):
        plt.plot(gs['date'], gs['volume'], label=shift, linewidth=2)
    plt.title(f"Daily Consult Volume — {facility}")
    plt.xlabel("Date")
    plt.ylabel("Consults")
    plt.legend(title="Shift")
    plt.tight_layout()
    out_path = os.path.join("figs", f"{facility}_consult_volume.png")
    plt.savefig(out_path, dpi=150)
    plt.close()

print("Saved facility line charts to ./figs/")

Saved facility line charts to ./figs/


In [ ]:
# path to your file
CSV_PATH = "output/synthetic_data.csv"   # <-- change if needed

# --- load & reshape ---
df = pd.read_csv(CSV_PATH, parse_dates=["date"], index_col=0)
display(df)

,facility,date,md1,md2,pm
0,AHG,2025-01-01,1.0,2.0,2.0
1,AHG,2025-01-02,3.0,6.0,0.0
2,AHG,2025-01-03,0.0,3.0,0.0
3,AHG,2025-01-04,3.0,8.0,0.0
4,AHG,2025-01-05,0.0,4.0,0.0
...,...,...,...,...,...
44160,NHG,2025-12-27,1.0,0.0,2.0
44161,NHG,2025-12-28,0.0,0.0,3.0
44162,NHG,2025-12-29,0.0,0.0,0.0
44163,NHG,2025-12-30,0.0,0.0,6.0


In [24]:
shift_cols = [c for c in ["md1", "md2", "pm"] if c in df.columns]

long = (
    long
    .groupby(["facility", "shift", "date"], as_index=False, sort=False)["volume"]
    .sum()
    .sort_values(["facility", "shift", "date"])
)

display(long)

,facility,shift,date,volume
0,AHG,md1,2025-01-01,1.0
1,AHG,md1,2025-01-02,3.0
2,AHG,md1,2025-01-03,0.0
3,AHG,md1,2025-01-04,3.0
4,AHG,md1,2025-01-05,0.0
...,...,...,...,...
130300,wplex,pm,2025-12-27,0.0
130301,wplex,pm,2025-12-28,0.0
130302,wplex,pm,2025-12-29,0.0
130303,wplex,pm,2025-12-30,0.0


In [ ]:
long.to_csv("output/synthetic_data_long.csv")

In [25]:
# --- helper: robust 31-day forecast for one series ---
def forecast_series(series: pd.Series, horizon=31):
    """
    series: pd.Series indexed by daily DatetimeIndex, numeric volumes (can have NaNs).
    Returns a pd.Series of length 'horizon' indexed by next dates.
    Strategy:
      1) Try ETS with weekly seasonality (7).
      2) If not enough data or fit fails, use weekday averages.
      3) If that fails, use last observed value (or 0).
      4) Clip to >=0 and round to integers.
    """
    # ensure daily frequency index (fill missing dates as NaN)
    s = series.asfreq("D")

    last_date = s.index.max()
    future_idx = pd.date_range(last_date + pd.Timedelta(days=1), periods=horizon, freq="D")

    # enough non-null points to even try modeling?
    s_nonnull = s.dropna()
    if len(s_nonnull) >= 28:  # ~4 weeks minimum for weekly seasonality
        try:
            model = ExponentialSmoothing(
                s_nonnull, trend=None, seasonal="add", seasonal_periods=7,
                initialization_method="estimated"
            ).fit(optimized=True, use_brute=True)
            fc = model.forecast(horizon)
        except Exception:
            fc = None
    else:
        fc = None

    # fallback 1: weekday means (computed on most recent 8 weeks if available)
    if fc is None:
        # optional: focus on recent history to adapt to drift
        recent_window = 8 * 7
        s_recent = s_nonnull.iloc[-recent_window:] if len(s_nonnull) >= 14 else s_nonnull

        if len(s_recent) >= 7:
            by_wd = s_recent.groupby(s_recent.index.dayofweek).mean()  # 0=Mon ... 6=Sun
            # if some weekdays missing, fill them with the overall mean
            overall_mean = s_recent.mean() if len(s_recent) else 0.0
            by_wd = by_wd.reindex(range(7)).fillna(overall_mean)
            fc = pd.Series([by_wd[d.weekday()] for d in future_idx], index=future_idx)
        else:
            fc = None

    # fallback 2: last observed value (or zero)
    if fc is None:
        last_val = s_nonnull.iloc[-1] if len(s_nonnull) else 0.0
        fc = pd.Series([last_val] * horizon, index=future_idx)

    # post-process: no negatives, round to integers (volumes are counts)
    fc = fc.clip(lower=0)
    fc = fc.round().astype(int)
    fc.index.name = "date"
    return fc

# --- run per (facility, shift) ---
forecasts = []
for (fac, sh), g in long.groupby(["facility", "shift"], sort=False):
    # build a daily series indexed by date
    s = g.set_index("date")["volume"].sort_index()
    fc = forecast_series(s, horizon=31)
    tmp = (
        fc.reset_index()
          .assign(facility=fac, shift=sh)
          .rename(columns={0: "forecast"})
    )
    tmp["forecast"] = tmp[sh if sh in shift_cols else "forecast"] if sh in tmp.columns else tmp["forecast"]
    tmp = tmp.rename(columns={fc.name if fc.name else "volume": "forecast"})  # guard
    tmp = tmp[["facility", "shift", "date", "forecast"]]
    forecasts.append(tmp)

forecast_long = pd.concat(forecasts, ignore_index=True)

In [ ]:
forecast_long.to_csv('output/forecast_long.csv')

In [26]:
display(forecast_long)

,facility,shift,date,forecast
0,AHG,md1,2026-01-01,1
1,AHG,md1,2026-01-02,1
2,AHG,md1,2026-01-03,1
3,AHG,md1,2026-01-04,1
4,AHG,md1,2026-01-05,1
...,...,...,...,...
11062,wplex,pm,2026-01-27,0
11063,wplex,pm,2026-01-28,0
11064,wplex,pm,2026-01-29,0
11065,wplex,pm,2026-01-30,0


In [28]:
from pathlib import Path
import matplotlib.pyplot as plt

In [29]:
# Safety: ensure datetime
long = long.copy()
long["date"] = pd.to_datetime(long["date"])
forecast_long = forecast_long.copy()
forecast_long["date"] = pd.to_datetime(forecast_long["date"])

# Output dirs/files
plot_dir = Path("plots")
plot_dir.mkdir(exist_ok=True)
pdf_path = plot_dir / "all_facility_shift_forecasts.pdf"

In [31]:
def plot_one_series(fac, sh, g_hist, g_fore):
    # Prepare series (sorted, dropna)
    s_hist = g_hist.sort_values("date")[["date","volume"]].dropna()
    s_fore = g_fore.sort_values("date")[["date","forecast"]].dropna()

    # Create figure (one chart per figure; no seaborn; no explicit colors)
    plt.figure(figsize=(9, 4.5))
    plt.plot(s_hist["date"], s_hist["volume"], label="history")
    plt.plot(s_fore["date"], s_fore["forecast"], linestyle="--", label="forecast (31d)")

    # Aesthetics
    plt.title(f"{fac} – {sh} forecast")
    plt.xlabel("Date")
    plt.ylabel("Patient volume")
    plt.grid(True, which="both", linestyle=":", linewidth=0.7)
    plt.legend()

    # Save PNG
    fname = plot_dir / f"{fac}_{sh}_forecast.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150)

    # Close the figure to free memory when looping many charts
    plt.close()

# Loop over all facility/shift pairs present in either hist or forecast
pairs = (
    pd.concat([
        long[["facility","shift"]].drop_duplicates(),
        forecast_long[["facility","shift"]].drop_duplicates()
    ]).drop_duplicates().sort_values(["facility","shift"])
)

for _, row in pairs.iterrows():
    fac, sh = row["facility"], row["shift"]
    g_hist = long[(long["facility"] == fac) & (long["shift"] == sh)]
    g_fore = forecast_long[(forecast_long["facility"] == fac) & (forecast_long["shift"] == sh)]

    # If both empty (shouldn't happen), skip
    if g_hist.empty and g_fore.empty:
        continue

    plot_one_series(fac, sh, g_hist, g_fore)